Gold Table: Edits per Wiki
This helps the dashboard show which Wikipedia site has the most activity.

In [0]:
%sql
SELECT COUNT(*)
FROM real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS real_time_wikipedia_streaming_pipeline.gold;

In [0]:
%sql
CREATE OR REPLACE TABLE real_time_wikipedia_streaming_pipeline.gold.wiki_edit_counts AS
SELECT
    wiki,
    COUNT(*) AS total_edits,
    COUNT(CASE WHEN bot = true THEN 1 END) AS bot_edits,
    COUNT(CASE WHEN bot = false THEN 1 END) AS human_edits
FROM real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw
GROUP BY wiki;

In [0]:
%sql
CREATE OR REPLACE TABLE real_time_wikipedia_streaming_pipeline.gold.top_users AS
SELECT
    user,
    COUNT(*) AS total_edits
FROM real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw
WHERE user IS NOT NULL
GROUP BY user
ORDER BY total_edits DESC
LIMIT 20;

In [0]:
%sql
CREATE OR REPLACE TABLE real_time_wikipedia_streaming_pipeline.gold.namespace_activity AS
SELECT
    namespace,
    COUNT(*) AS edit_count
FROM real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw
GROUP BY namespace
ORDER BY edit_count DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE real_time_wikipedia_streaming_pipeline.gold.edits_over_time AS
SELECT
    date_trunc('minute', to_timestamp(CAST(timestamp AS LONG))) AS edit_minute,
    COUNT(*) AS edits
FROM real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw
GROUP BY edit_minute
ORDER BY edit_minute;

In [0]:
%sql
CREATE OR REPLACE TABLE real_time_wikipedia_streaming_pipeline.gold.top_pages AS
SELECT
    title,
    COUNT(*) AS edit_count
FROM real_time_wikipedia_streaming_pipeline.silver.real_time_wikipedia_streaming_data_vw
GROUP BY title
ORDER BY edit_count DESC
LIMIT 20;